# Evaluate BioASQ factoid generation methods

This notebook lets you choose any local base model + optional LoRA adapter, choose dev or test split, and compare three inference methods with the official BioASQ factoid metrics:

1. `single_generation`: one greedy generation, evaluated as one answer.
2. `sample10_frequency`: sample 10 candidates, normalize single-answer candidates, rank by frequency, and evaluate the ranked top-5 list.
3. `sample5_top5`: sample 5 candidates and evaluate the unique top-5 list in generation order.

Outputs are written under `Artifacts/cse_dpo/generation_method_evals/<run_name>/`.


In [ ]:
import gc
import json
import os
import re
import sys
import tempfile
from collections import Counter
from datetime import datetime
from pathlib import Path
from typing import Any

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
os.environ.setdefault("HF_HUB_OFFLINE", "1")
os.environ.setdefault("TRANSFORMERS_OFFLINE", "1")
os.environ.setdefault("TRANSFORMERS_VERBOSITY", "error")

import numpy as np
import pandas as pd
import torch
from peft import PeftModel
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents, start / "Task-Structured Counterfactual Preference Mining"]:
        if (candidate / "cse_dpo").is_dir() and (candidate / "src").is_dir():
            return candidate.resolve()
    raise RuntimeError("Could not locate project root. Start this notebook in the project or workspace directory.")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from cse_dpo.generated_bioasq_eval import load_gold_examples
from cse_dpo.normalize_set_answers import normalize_answer_surface
from src.prompt_registry import resolve_prompt_bundle
from src.utility.bioasq_format import exact_answer_groups, parse_prediction_items
from src.utility.bioasq_official import evaluate_with_bioasq_java
from src.utility.config import QUESTION_INSTRUCTIONS
from src.utility.data import clean_text
from src.utility.eval_dataset import load_eval_examples, render_prompt
from src.utility.eval_types import EvalExample

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Project root:", PROJECT_ROOT)
print("Device:", torch.cuda.get_device_name(DEVICE) if DEVICE.type == "cuda" else "CPU")


## Configuration

Change only this cell for most runs. `MODEL_PRESETS` contains common local choices, but you can also set `BASE_MODEL_REF` and `ADAPTER_PATH` manually.

For `SPLIT`, use:

- `dev`: 160 BioASQ dev factoid questions from the prepared full-resource dev file.
- `test`: BioASQ Task13B test golden files if they are present locally.
- `custom`: set `CUSTOM_EVAL_FILES` yourself.


In [ ]:
MODEL_PRESETS = {
    "qwen25_05b_sft_strict_extractive": {
        "base_model": PROJECT_ROOT / "models/Qwen2.5-0.5B-Instruct",
        "adapter": PROJECT_ROOT / "Artifacts/Factoid_SFT/models/evidence_grounded_per_supported_alias_qwen25_05b_lora_dropout_005_strict_extractive/adapter",
        "prompt_ref": "factoid-single-answer-extractive-v1",
    },
    "qwen25_3b_sft_strict_extractive_best_mrr": {
        "base_model": PROJECT_ROOT / "models/Qwen2.5-3B-Instruct",
        "adapter": PROJECT_ROOT / "Artifacts/Factoid_SFT/models/evidence_grounded_per_supported_alias_qwen25_3b_lora_dropout_005_strict_extractive/adapter_best_evidence_mrr",
        "prompt_ref": "factoid-single-answer-extractive-v1",
    },
    "qwen25_3b_stage2_retry_05b_settings": {
        "base_model": PROJECT_ROOT / "models/Qwen2.5-3B-Instruct",
        "adapter": PROJECT_ROOT / "Artifacts/cse_dpo/three_stage_dpo_qwen25_3b_c3_c1_curriculum_semantic_full_bp2048/stage_2_format_alignment_retry_05b_stage2_settings/best_selected_adapter",
        "prompt_ref": "factoid-single-answer-extractive-v1",
    },
}

MODEL_PRESET = "qwen25_3b_sft_strict_extractive_best_mrr"
BASE_MODEL_REF = MODEL_PRESETS[MODEL_PRESET]["base_model"]
ADAPTER_PATH = MODEL_PRESETS[MODEL_PRESET]["adapter"]  # Set to None for base-model-only evaluation.
PROMPT_REF = MODEL_PRESETS[MODEL_PRESET]["prompt_ref"]

SPLIT = "dev"  # "dev", "test", or "custom"
CUSTOM_EVAL_FILES = []  # Only used when SPLIT == "custom".
LIMIT_QUESTIONS = None  # Use a small number like 5 for smoke testing.

MAX_SEQ_LENGTH = 4096  # Left-truncates the prompt during generation if needed.
MAX_NEW_TOKENS = 64
LOAD_IN_4BIT = True
LOCAL_FILES_ONLY = True

# Sampling settings for methods 2 and 3.
SAMPLING_TEMPERATURE = 0.7
SAMPLING_TOP_P = 0.9
SAMPLING_SEED = 3407

METHODS_TO_RUN = ["single_generation","sample5_top5"]
RUN_NAME = f"{MODEL_PRESET}_{SPLIT}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
OUTPUT_DIR = PROJECT_ROOT / "Artifacts/cse_dpo/generation_method_evals" / RUN_NAME
OUTPUT_DIR.mkdir(parents=True, exist_ok=False)

print(json.dumps({
    "model_preset": MODEL_PRESET,
    "base_model": str(BASE_MODEL_REF),
    "adapter": None if ADAPTER_PATH is None else str(ADAPTER_PATH),
    "prompt_ref": PROMPT_REF,
    "split": SPLIT,
    "output_dir": str(OUTPUT_DIR),
}, indent=2))


In [ ]:
def source_args(limit=None):
    return type("SourceArgs", (), {
        "question_types": ["factoid"],
        "max_summary_answers": 5,
        "max_factoid_answers": 5,
        "max_list_items": 100,
        "max_resources": 0,
        "max_resource_chars": 0,
        "resource_selection": "first",
        "resource_granularity": "document",
        "resource_window_mode": "single",
        "resource_reranker_model": None,
        "resource_reranker_article_model": None,
        "resource_reranker_device": None,
        "resource_reranker_batch_size": 1,
        "local_files_only": True,
        "limit": limit,
    })()


def eval_files_for_split(split: str) -> list[Path]:
    split = split.lower().strip()
    if split == "dev":
        return [PROJECT_ROOT / "data/BioASQ_factoid_sft_prepared/single_answer_full_resources_qwen25_05b/eval_prepared.json"]
    if split == "test":
        files = sorted((PROJECT_ROOT / "data/Task13BTest").glob("*_golden.json"))
        if not files:
            raise FileNotFoundError("No Task13B test golden files found under data/Task13BTest/*_golden.json")
        return files
    if split == "custom":
        files = [Path(x) for x in CUSTOM_EVAL_FILES]
        if not files:
            raise ValueError("CUSTOM_EVAL_FILES is empty.")
        return files
    raise ValueError(f"Unknown SPLIT={split!r}")


prompt_bundle = resolve_prompt_bundle(
    PROJECT_ROOT / "prompts/factoid_single_answer_aligned.json",
    PROMPT_REF,
    QUESTION_INSTRUCTIONS,
)
eval_files = eval_files_for_split(SPLIT)
examples = load_eval_examples(eval_files, source_args(LIMIT_QUESTIONS), prompt_bundle["instructions"])
gold_examples = load_gold_examples(eval_files)
examples = [example for example in examples if example.question_id in gold_examples]
if LIMIT_QUESTIONS is not None:
    examples = examples[: int(LIMIT_QUESTIONS)]

print("Prompt ID:", prompt_bundle["prompt_id"])
print("Eval files:")
for path in eval_files:
    print(" -", path)
print("Loaded factoid examples:", len(examples))
assert examples, "No factoid examples loaded."


In [ ]:
def load_model_and_tokenizer(base_model_ref: Path, adapter_path: Path | None):
    tokenizer_path = adapter_path if adapter_path is not None and (adapter_path / "tokenizer_config.json").exists() else base_model_ref
    tokenizer = AutoTokenizer.from_pretrained(str(tokenizer_path), local_files_only=LOCAL_FILES_ONLY)
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"

    quantization_config = None
    dtype = torch.float16 if DEVICE.type == "cuda" else torch.float32
    model_kwargs = {
        "low_cpu_mem_usage": True,
        "local_files_only": LOCAL_FILES_ONLY,
    }
    if DEVICE.type == "cuda":
        model_kwargs["device_map"] = {"": DEVICE.index or 0}
    if LOAD_IN_4BIT and DEVICE.type == "cuda":
        quantization_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=torch.float16,
        )
        model_kwargs["quantization_config"] = quantization_config
    else:
        model_kwargs["torch_dtype"] = dtype

    base_model = AutoModelForCausalLM.from_pretrained(str(base_model_ref), **model_kwargs)
    if adapter_path is not None:
        model = PeftModel.from_pretrained(base_model, str(adapter_path), local_files_only=LOCAL_FILES_ONLY)
    else:
        model = base_model
    model.config.use_cache = False
    model.eval()
    return model, tokenizer


def release_model(model=None):
    if model is not None:
        del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


model, tokenizer = load_model_and_tokenizer(Path(BASE_MODEL_REF), None if ADAPTER_PATH is None else Path(ADAPTER_PATH))
print(type(model).__name__)


In [ ]:
def write_json(path: Path, payload: Any):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")


def strip_answer_prefix(text: str) -> str:
    text = clean_text(text)
    return re.sub(r"^answer\s*:\s*", "", text, flags=re.IGNORECASE).strip()


def generate_one(prompt: str, *, do_sample: bool, seed: int | None = None) -> str:
    previous_truncation_side = getattr(tokenizer, "truncation_side", None)
    try:
        if previous_truncation_side is not None:
            tokenizer.truncation_side = "left"
        encoded = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=MAX_SEQ_LENGTH)
    finally:
        if previous_truncation_side is not None:
            tokenizer.truncation_side = previous_truncation_side

    device = next(model.parameters()).device
    encoded = {k: v.to(device) for k, v in encoded.items()}
    generation_kwargs = {
        "max_new_tokens": MAX_NEW_TOKENS,
        "pad_token_id": tokenizer.pad_token_id,
        "eos_token_id": tokenizer.eos_token_id,
        "do_sample": bool(do_sample),
        "use_cache": False,
    }
    if do_sample:
        generation_kwargs.update({"temperature": SAMPLING_TEMPERATURE, "top_p": SAMPLING_TOP_P})
    if seed is not None:
        torch.manual_seed(int(seed))
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(int(seed))
    with torch.inference_mode():
        output_ids = model.generate(**encoded, **generation_kwargs)
    prompt_len = encoded["input_ids"].shape[-1]
    decoded = tokenizer.decode(output_ids[0][prompt_len:], skip_special_tokens=True)
    del encoded, output_ids
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return strip_answer_prefix(decoded)


def extract_single_candidate(sample: str) -> str | None:
    items = parse_prediction_items(sample, "factoid")
    if len(items) != 1:
        return None
    candidate = clean_text(items[0])
    return candidate or None


def normalize_candidate(candidate: str) -> str:
    return normalize_answer_surface(candidate) or clean_text(candidate).casefold()


def format_factoid_prediction(candidates: list[str]) -> str:
    cleaned = []
    seen = set()
    for candidate in candidates:
        candidate = clean_text(candidate)
        key = normalize_candidate(candidate)
        if not candidate or not key or key in seen:
            continue
        seen.add(key)
        cleaned.append(candidate)
    return " ".join(f"[BE] {candidate} [EE]" for candidate in cleaned[:5])


def rank_by_frequency(samples: list[str], *, max_answers: int = 5) -> tuple[list[str], dict[str, Any]]:
    votes = Counter()
    representative = {}
    first_seen = {}
    invalid_samples = []
    for i, sample in enumerate(samples):
        candidate = extract_single_candidate(sample)
        if candidate is None:
            invalid_samples.append({"sample_index": i, "sample": sample})
            continue
        key = normalize_candidate(candidate)
        if not key:
            invalid_samples.append({"sample_index": i, "sample": sample})
            continue
        votes[key] += 1
        representative.setdefault(key, candidate)
        first_seen.setdefault(key, i)
    ranked_keys = sorted(votes, key=lambda key: (-votes[key], first_seen[key]))
    ranked = [representative[key] for key in ranked_keys[:max_answers]]
    if not ranked and samples:
        fallback = extract_single_candidate(samples[0]) or strip_answer_prefix(samples[0])
        ranked = [fallback] if fallback else []
    return ranked, {
        "valid_single_answer_sample_count": int(sum(votes.values())),
        "unique_valid_candidate_count": int(len(votes)),
        "invalid_sample_count": int(len(invalid_samples)),
        "votes": {representative[key]: int(votes[key]) for key in ranked_keys},
        "invalid_samples": invalid_samples,
    }


def rank_by_generation_order(samples: list[str], *, max_answers: int = 5) -> tuple[list[str], dict[str, Any]]:
    ranked = []
    seen = set()
    invalid_samples = []
    for i, sample in enumerate(samples):
        candidate = extract_single_candidate(sample)
        if candidate is None:
            invalid_samples.append({"sample_index": i, "sample": sample})
            continue
        key = normalize_candidate(candidate)
        if not key or key in seen:
            continue
        seen.add(key)
        ranked.append(candidate)
        if len(ranked) >= max_answers:
            break
    if not ranked and samples:
        fallback = extract_single_candidate(samples[0]) or strip_answer_prefix(samples[0])
        ranked = [fallback] if fallback else []
    return ranked, {
        "valid_single_answer_sample_count": len([s for s in samples if extract_single_candidate(s) is not None]),
        "unique_valid_candidate_count": len(seen),
        "invalid_sample_count": len(invalid_samples),
        "invalid_samples": invalid_samples,
    }


In [ ]:
def official_score_args():
    return type("OfficialArgs", (), {
        "bioasq_java_jar": str(PROJECT_ROOT / "third_party/Evaluation-Measures/flat/BioASQEvaluation/dist/BioASQEvaluation.jar"),
        "bioasq_java_heap": "512m",
        "bioasq_java_version": 5,
    })()


def evaluate_prediction_rows(prediction_rows: list[dict[str, Any]], method_dir: Path, method_name: str):
    examples_by_key = {
        (row["question_id"], row["question_type"]): gold_examples[row["question_id"]]
        for row in prediction_rows
    }
    official = evaluate_with_bioasq_java(
        prediction_rows=prediction_rows,
        examples_by_key=examples_by_key,
        model_label=f"{MODEL_PRESET}-{method_name}",
        model_dir=method_dir,
        args=official_score_args(),
        include_per_question=True,
    )
    official_by_id = {row["question_id"]: row for row in official["per_question"]}
    enriched = []
    for row in prediction_rows:
        merged = dict(row)
        merged.update({k: v for k, v in official_by_id[row["question_id"]].items() if k not in {"question_id", "question_type"}})
        enriched.append(merged)
    factoid_metrics = official["aggregate"].get("by_type", {}).get("factoid", {}).get("metrics", {})
    summary = {
        "method": method_name,
        "question_count": len(enriched),
        "mrr": factoid_metrics.get("mrr"),
        "strict_accuracy": factoid_metrics.get("strict_accuracy"),
        "lenient_accuracy": factoid_metrics.get("lenient_accuracy"),
        "official_scores_path": str(method_dir / "official_bioasq" / "official_scores.json"),
    }
    write_json(method_dir / "summary.json", summary)
    write_json(method_dir / "per_question_metrics.json", enriched)
    pd.DataFrame(enriched).to_csv(method_dir / "per_question_metrics.csv", index=False)
    return summary, enriched


def run_method(method_name: str):
    method_dir = OUTPUT_DIR / method_name
    method_dir.mkdir(parents=True, exist_ok=False)
    rows = []
    input_records = []
    for row_index, example in enumerate(tqdm(examples, desc=method_name)):
        prompt = render_prompt(tokenizer, example, chat_template=prompt_bundle.get("chat_template", "qwen-2.5"), prompt_format="chat")
        input_records.append({
            "question_id": example.question_id,
            "question": example.body,
            "gold_output": example.gold_output,
            "resource_count": len(example.resources),
            "prompt": prompt,
        })
        if method_name == "single_generation":
            sample = generate_one(prompt, do_sample=False)
            candidate = extract_single_candidate(sample) or sample
            ranked_candidates = [candidate] if candidate else []
            diagnostics = {
                "generation_samples": [sample],
                "valid_single_answer_sample_count": 1 if extract_single_candidate(sample) is not None else 0,
                "unique_valid_candidate_count": 1 if extract_single_candidate(sample) is not None else 0,
                "invalid_sample_count": 0 if extract_single_candidate(sample) is not None else 1,
            }
        elif method_name == "sample10_frequency":
            samples = [generate_one(prompt, do_sample=True, seed=SAMPLING_SEED + row_index * 10 + i) for i in range(10)]
            ranked_candidates, diagnostics = rank_by_frequency(samples, max_answers=5)
            diagnostics["generation_samples"] = samples
        elif method_name == "sample5_top5":
            samples = [generate_one(prompt, do_sample=True, seed=SAMPLING_SEED + row_index * 5 + i) for i in range(5)]
            ranked_candidates, diagnostics = rank_by_generation_order(samples, max_answers=5)
            diagnostics["generation_samples"] = samples
        else:
            raise ValueError(f"Unknown method: {method_name}")

        prediction = format_factoid_prediction(ranked_candidates)
        rows.append({
            "question_id": example.question_id,
            "question_type": example.question_type,
            "body": example.body,
            "source_path": example.source_path,
            "prediction": prediction,
            "ranked_candidates": ranked_candidates,
            "prediction_count": len(parse_prediction_items(prediction, example.question_type)),
            "gold_output": example.gold_output,
            "gold_count": len(exact_answer_groups(example, example.question_type)),
            **diagnostics,
        })
    write_json(method_dir / "evaluation_inputs.json", input_records)
    return evaluate_prediction_rows(rows, method_dir, method_name)


all_summaries = []
for method_name in METHODS_TO_RUN:
    summary, _ = run_method(method_name)
    all_summaries.append(summary)
    print(json.dumps(summary, indent=2))

summary_df = pd.DataFrame(all_summaries)
summary_df.to_csv(OUTPUT_DIR / "method_summary.csv", index=False)
write_json(OUTPUT_DIR / "method_summary.json", all_summaries)
summary_df


In [ ]:
# Optional: inspect overlap of correct/wrong questions across methods.
method_frames = {}
for method_name in METHODS_TO_RUN:
    path = OUTPUT_DIR / method_name / "per_question_metrics.csv"
    if path.exists():
        df = pd.read_csv(path)
        df["is_correct_mrr"] = df["mrr"].fillna(0).astype(float) > 0
        method_frames[method_name] = df[["question_id", "prediction", "mrr", "strict_accuracy", "lenient_accuracy", "is_correct_mrr"]]

if method_frames:
    overlap = None
    for method_name, df in method_frames.items():
        view = df.rename(columns={
            "prediction": f"{method_name}_prediction",
            "mrr": f"{method_name}_mrr",
            "strict_accuracy": f"{method_name}_strict",
            "lenient_accuracy": f"{method_name}_lenient",
            "is_correct_mrr": f"{method_name}_correct",
        })
        overlap = view if overlap is None else overlap.merge(view, on="question_id", how="outer")
    overlap.to_csv(OUTPUT_DIR / "method_question_overlap.csv", index=False)
    display(overlap.head())
    print("Saved:", OUTPUT_DIR / "method_question_overlap.csv")


In [ ]:
# Optional cleanup when you finish the notebook.
# release_model(model)
